In [7]:
import re
from kyx_example_content import KYX_EXAMPLE_CONTENT
import os

In [16]:
class ASTNode:
    """Base class for all nodes in the Abstract Syntax Tree."""
    def __init__(self, node_type, value=None, children=None):
        self.node_type = node_type
        self.value = value
        self.children = children if children is not None else []

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self, level=0):
        """Generates a readable, indented representation of the tree."""
        indent = "  " * level
        value_str = f": {self.value}" if self.value is not None else ""
        s = f"{indent}[{self.node_type}]{value_str}\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class Definition(ASTNode):
    """Represents a constant, function, or predicate definition."""
    def __init__(self, name, definition_type, value=None, comment=None):
        # Store name in value for base ASTNode representation
        super().__init__(f"{definition_type}Definition", name) 
        self.name = name
        self.definition_type = definition_type
        self.definition_value = value
        self.comment = comment

    def __repr__(self, level=0):
        indent = "  " * level
        # Use definition_value for display, as it holds the expression body
        value_str = f" = {' '.join(self.definition_value.split())[:60]}..." if self.definition_value else ""
        comment_str = f" /* {self.comment} */" if self.comment else ""
        s = f"{indent}[DEF] {self.definition_type} {self.name}{value_str}{comment_str}\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class HybridProgram(Definition):
    """
    Represents a Hybrid Program (HP) definition. 
    It parses the internal structure (choices, sequences, ODEs).
    """
    def __init__(self, name, body):
        # Pass the body as the definition value
        super().__init__(name, "HP", value=body) 
        self.body = body
        self._parse_hp_structure() # Parse structure immediately

    def _parse_hp_structure(self):
        body_content = self.body.strip()
        
        if self.name == 'controlFunc':
            # Parse Choice Operator (++)
            choice_node = ASTNode("ChoiceOperator (++)")
            self.add_child(choice_node)
            
            # Split blocks based on '++'
            blocks = re.split(r'\s*\+\+\s*', body_content)
            
            for i, block in enumerate(blocks):
                block = block.strip()
                # Remove surrounding braces { ... }
                if block.startswith('{') and block.endswith('}'):
                    block = block[1:-1].strip()
                if block:
                    choice_node.add_child(ASTNode(f"Choice {i+1}", block))

        elif self.name == 'stepFunction':
            # Parse ODEs and Evolution Domain
            
            # Clean up the body: remove newlines/extra spaces
            clean_body = re.sub(r'\s+', ' ', body_content).strip()
            
            # Split by comma
            parts = [p.strip() for p in clean_body.split(',')]
            
            ode_node = ASTNode("OrdinaryDifferentialEquations")
            domain_node = ASTNode("EvolutionDomain")
            
            for part in parts:
                if not part: continue
                
                # Check for evolution domain constraints (usually attached to t')
                if '&' in part:
                    # Split the ODE part from the domain constraints
                    sub_parts = [sp.strip() for sp in part.split('&')]
                    
                    # The first part is the ODE (e.g., t' = 1)
                    ode_node.add_child(ASTNode("ODE", sub_parts[0]))
                    
                    # The remaining parts are the domain constraints
                    for domain_constraint in sub_parts[1:]:
                        domain_node.add_child(ASTNode("Constraint", domain_constraint))
                else:
                    # Standard ODE
                    ode_node.add_child(ASTNode("ODE", part))
            
            self.add_child(ode_node)
            if domain_node.children:
                self.add_child(domain_node)

    def __repr__(self, level=0):
        indent = "  " * level
        s = f"{indent}[HP] {self.name}\n"
        s += f"{indent}  Body (raw): {' '.join(self.body.split())[:80]}...\n"
        
        # Print children (parsed structure)
        for child in self.children:
            s += child.__repr__(level + 1)
        return s

class Variable(ASTNode):
    """Represents a variable declaration in the ProgramVariables block."""
    def __init__(self, name, var_type, comment=None):
        super().__init__("Variable", name)
        self.var_type = var_type
        self.comment = comment

    def __repr__(self, level=0):
        indent = "  " * level
        comment_str = f" /* {self.comment} */" if self.comment else ""
        return f"{indent}[VAR] {self.var_type} {self.value}{comment_str}\n"

class ProblemStatement(ASTNode):
    """Represents the main Problem formula (P -> [HP] Q)."""
    def __init__(self, formula):
        super().__init__("ProblemStatement")
        self.formula = formula

    def __repr__(self, level=0):
        indent = "  " * level
        s = f"{indent}[PROBLEM]\n"
        for child in self.children:
            s += child.__repr__(level + 1)
        return s


# --- Parser Implementation ---

class KeYmaeraXParser:
    def __init__(self, source_code):
        self.source_code = source_code
        self.ast_root = ASTNode("KeYmaeraXSpecification")
        self._cleanup_source()

    def _cleanup_source(self):
        # 1. Remove multi-line comments (/* ... */)
        self.source_code = re.sub(r'/\*.*?\*/', lambda m: ' ' * len(m.group(0)), self.source_code, flags=re.DOTALL)
        # 2. Remove single-line comments (//)
        self.source_code = re.sub(r'//.*', '', self.source_code)
        # 3. Normalize line endings and remove excessive blank lines
        self.source_code = re.sub(r'\n\s*\n', '\n', self.source_code)
        self.source_code = self.source_code.strip()

    def parse(self):
        self._parse_theorem_name()
        self._parse_definitions()
        self._parse_program_variables()
        self._parse_problem()
        return self.ast_root

    def _parse_theorem_name(self):
        match = re.search(r'Theorem "(.*?)"', self.source_code)
        if match:
            self.ast_root.add_child(ASTNode("TheoremName", match.group(1)))

    def _extract_block(self, keyword):
        # Extracts content between 'Keyword' and 'End.' or the next major keyword block
        pattern = rf'{keyword}\s*(.*?)(?=\n\s*End\.|\n\s*Theorem|\n\s*Definitions|\n\s*ProgramVariables|\n\s*Problem|$)'
        match = re.search(pattern, self.source_code, re.DOTALL | re.IGNORECASE)
        if match:
            content = match.group(1).strip()
            if content.endswith('End.'):
                 content = content[:-4].strip()
            return content
        return None

    def _parse_definitions(self):
        definitions_block = self._extract_block("Definitions")
        if not definitions_block:
            return

        defs_node = ASTNode("DefinitionsBlock")
        self.ast_root.add_child(defs_node)

        # Split by ;; or ;
        parts = re.split(r'(?:;;|;)', definitions_block)
        
        for part in parts:
            line = part.strip()
            if not line:
                continue

            # 1. Parse HP Definitions (Hybrid Programs)
            # FIX: Allow optional trailing semicolons (;) after the closing brace }
            hp_match = re.search(r'HP\s+(\w+)\s*::=\s*\{(.*?)\}\s*;*\s*$', line, re.DOTALL)
            if hp_match:
                name = hp_match.group(1).strip()
                body = hp_match.group(2).strip()
                defs_node.add_child(HybridProgram(name, body))
                continue

            # 2. Parse Real/Bool Definitions
            def_match = re.search(r'(Real|Bool)\s+(\w+)(?:\((.*?)\))?\s*(?:<->|=)\s*(.*?)$', line, re.DOTALL)
            
            if def_match:
                def_type = def_match.group(1)
                name = def_match.group(2)
                params = def_match.group(3)
                value = def_match.group(4).strip()
                
                if value.startswith('(') and value.endswith(')'):
                    value = value[1:-1].strip()
                
                def_node = Definition(name, def_type, value=value)
                if params:
                    def_node.add_child(ASTNode("Parameters", params))
                
                defs_node.add_child(def_node)
                continue
            
            # 3. Parse simple constant declarations
            const_match = re.search(r'(Real|Bool)\s+(\w+)\s*$', line, re.DOTALL)
            if const_match:
                def_type = const_match.group(1)
                name = const_match.group(2)
                defs_node.add_child(Definition(name, def_type))


    def _parse_program_variables(self):
        vars_block = self._extract_block("ProgramVariables")
        if not vars_block:
            return

        vars_node = ASTNode("ProgramVariablesBlock")
        self.ast_root.add_child(vars_node)

        for line in vars_block.split(';'):
            line = line.strip()
            if not line:
                continue
            
            var_match = re.search(r'(Real|Bool)\s+(\w+)\s*$', line, re.DOTALL)
            if var_match:
                var_type = var_match.group(1)
                name = var_match.group(2)
                vars_node.add_child(Variable(name, var_type))

    def _parse_problem(self):
        problem_block = self._extract_block("Problem")
        if not problem_block:
            return

        problem_node = ProblemStatement(problem_block)
        self.ast_root.add_child(problem_node)

        # Detailed parsing of the implication structure: P -> [HP] Q
        implication_match = re.search(r'(.*?)->\s*[(.*?)]\s*(.*)', problem_block, re.DOTALL)
        
        if implication_match:
            precondition_raw = implication_match.group(1).strip()
            program_raw = implication_match.group(2).strip()
            postcondition_raw = implication_match.group(3).strip()

            problem_node.add_child(ASTNode("Precondition (P)", precondition_raw))
            
            # Parse the loop structure: { ... }*@invariant(loopinv)
            loop_match = re.search(r'\{(.*?)\}\s*\*\s*@invariant\((.*?)\)', program_raw, re.DOTALL)
            
            if loop_match:
                loop_body_raw = loop_match.group(1).strip()
                loop_invariant_raw = loop_match.group(2).strip()
                
                program_node = ASTNode("LoopExecution")
                program_node.add_child(ASTNode("LoopInvariant (I)", loop_invariant_raw))
                
                inner_blocks = re.findall(r'\{(.*?)\}', loop_body_raw, re.DOTALL)
                
                if len(inner_blocks) >= 2:
                    
                    control_block_content = inner_blocks[0].strip()
                    program_node.add_child(ASTNode("ControlBlock (Discrete)", control_block_content))
                    
                    dynamics_block_content = inner_blocks[1].strip()
                    dynamics_node = ASTNode("DynamicsBlock (Continuous)")
                    
                    ode_inv_match = re.search(r'@invariant\((.*?)\)', dynamics_block_content, re.DOTALL)
                    if ode_inv_match:
                        ode_invariants_raw = ode_inv_match.group(1).strip()
                        dynamics_node.add_child(ASTNode("ODE Invariants", ode_invariants_raw))
                        
                        core_dynamics = dynamics_block_content.replace(ode_inv_match.group(0), '').strip()
                        dynamics_node.add_child(ASTNode("HP Call", core_dynamics))
                    else:
                         dynamics_node.add_child(ASTNode("HP Call", dynamics_block_content))
                         
                    program_node.add_child(dynamics_node)
                
                problem_node.add_child(program_node)
            else:
                problem_node.add_child(ASTNode("RawProgram", program_raw))

            problem_node.add_child(ASTNode("Postcondition (Q)", postcondition_raw))
        else:
            problem_node.add_child(ASTNode("Error", "Could not parse P -> [HP] Q structure."))


In [17]:
# --- File Loading and Execution ---

def load_kyx_file(filepath):
    """Loads the content of a KYX file."""
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None
    except Exception as e:
        print(f"An error occurred while reading the file: {e}")
        return None

def setup_example_file(filepath, content):
    """Utility function to create the example file if it doesn't exist."""
    if not os.path.exists(filepath):
        print(f"Creating example file: {filepath}")
        with open(filepath, 'w') as f:
            f.write(content)
    else:
        print(f"Using existing file: {filepath}")


# --- Main Execution ---

if __name__ == "__main__":
    # FILE_PATH = "D:\Reinforcement learning\kyx_parser\kyxs\CoastingCar_with_pred.kyx"
    FILE_PATH = "example.kyx"
    # Content used for creating the file if it doesn't exist
    
    
    # 1. Ensure the file exists
    setup_example_file(FILE_PATH, KYX_EXAMPLE_CONTENT)

    # 2. Load the source code from the file
    source_code = load_kyx_file(FILE_PATH)

    if source_code:
        # 3. Parse the loaded source code
        parser = KeYmaeraXParser(source_code)
        ast = parser.parse()
        
        # 4. Display the resulting AST
        print("\n--- Abstract Syntax Tree ---")
        print(ast)

Using existing file: example.kyx

--- Abstract Syntax Tree ---
[KeYmaeraXSpecification]
  [TheoremName]: IJRR17/Theorem 1: Static safety
  [DefinitionsBlock]
    [DEF] Real ep
    [DEF] Real b
    [DEF] Real A
    [DEF] Real W
    [DEF] Real stopDist = v^2 / (2*b)...
      [Parameters]: Real v
    [DEF] Real accelComp = (A/b + 1) * (A/2 * ep^2 + ep*v)...
      [Parameters]: Real v
    [DEF] Real admissibleSeparation = stopDist(v) + accelComp(v)...
      [Parameters]: Real v
    [DEF] Bool isWellformedDir = dx^2 + dy^2 = 1...
      [Parameters]: Real dx, Real dy
    [DEF] Bool bounds = A >= 0 & b > 0 & ep > 0...
    [HP] stepFunction
      Body (raw): x' = v * dx, y' = v * dy, v' = a, dx' = -w * dy, dy' = w * dx, w' = a/r, t' = 1 ...
      [OrdinaryDifferentialEquations]
        [ODE]: x' = v * dx
        [ODE]: y' = v * dy
        [ODE]: v' = a
        [ODE]: dx' = -w * dy
        [ODE]: dy' = w * dx
        [ODE]: w' = a/r
        [ODE]: t' = 1
      [EvolutionDomain]
        [Constra